# <center> <font color="#0036a3">Maestría en Inteligencia Artificial Aplicada (MNA)</font>  — New Version</center>

<center>

[![Materia](https://img.shields.io/badge/MATERIA-PROYECTO_INTEGRADOR-E0A800?style=for-the-badge&logoColor=white)](https://tec.mx)

</center>

<center>

[![Python](https://img.shields.io/badge/Python-3776AB?style=flat-square&logo=python&logoColor=white)](https://www.python.org/)
[![PyTorch](https://img.shields.io/badge/PyTorch-EE4C2C?style=flat-square&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![GitHub](https://img.shields.io/badge/Repo-GitHub-181717?style=flat-square&logo=github&logoColor=white)](https://github.com/jmtoral/proyecto_integrador_52)

</center>

## **<font color="#0036a3">Avance 5 — Evaluación de Image Enhancement sobre Modelos de Profundidad Endoscópica Entrenados en SCARED</font>**

### **<font color="#E0A800">Proyecto Integrador — TC5035.10</font>**

---

## **<center> <font color="#0036a3">Equipo 52</font> </center>**

<table style="border-collapse:collapse; width:60%; margin:auto;">
  <tr>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/elda.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>Elda Morales</strong><br><small>A00449074</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/mpgc.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>María Paula Gutiérrez</strong><br><small>A01747706</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/jmtc_n.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>José Manuel Toral</strong><br><small>A01122243</small>
    </td>
  </tr>
</table>

---

### Diferencia respecto a Avance 4

En el **Avance 4** se evaluaron modelos con sus pesos **originales del paper** (entrenados en Hamlyn, KITTI o Endovis).

En este **Avance 5** se usan pesos **re-entrenados en SCARED** (proporcionados por el Dr. Ricardo Espinosa Loera), evaluados sobre el **split oficial de AF-SfMLearner** (550 fotogramas, datasets 1–7). Esto responde la pregunta: **¿cambia el impacto del enhancement cuando el modelo conoce el dominio de evaluación?**

### Hipótesis
El fine-tuning en SCARED mejora el baseline (none) de todos los modelos. El efecto del enhancement se reduce para los modelos que ya manejan bien las especularidades del dominio.


## New Version — Evaluación con el *split* oficial y Endo-STTN

> **Equipo 52 · Proyecto Integrador TC5035.10 — MNA, Tecnológico de Monterrey**

Esta **New Version** del Avance 5 introduce dos cambios metodológicos importantes en la evaluación:

1. **Protocolo de prueba estándar.** En lugar de evaluar sobre 10 *keyframes* sueltos (datasets 8–9), ahora usamos el ***split* oficial de AF-SfMLearner** (`splits/endovis/test_files.txt`): **550 fotogramas** de los *datasets* 1–7 de SCARED. Este es el mismo conjunto de prueba que reportan los artículos del estado del arte, de modo que nuestros números son **directamente comparables con la literatura**.

2. **Endo-STTN como nuevo método de mejora.** Añadimos un quinto *enhancement*: **Endo-STTN** (*Spatio-Temporal Transformer Network*), un modelo de *inpainting* de vídeo que elimina los **reflejos especulares** aprovechando la información **temporal** de los fotogramas vecinos.

### ¿Qué pregunta de investigación respondemos?

> *¿Los métodos de realce de imagen (image enhancement) mejoran la estimación de profundidad monocular en endoscopía?*

Comparamos, para **6 modelos de profundidad entrenados en SCARED** (pesos del Dr. Ricardo Espinosa Loera) y **5 métodos de realce**, el error de profundidad respecto al *ground-truth* estéreo de SCARED.

### Flujo general del experimento

```mermaid
flowchart LR
    A[SCARED zip<br/>rgb.mp4 + scene_points] --> B[Extracción<br/>550 frames del split]
    B --> C{Enhancement}
    C -->|none| D[Modelo de<br/>profundidad]
    C -->|retinex| D
    C -->|endolmspec| D
    C -->|iat| D
    C -->|endosttn| D
    D --> E[Profundidad<br/>predicha]
    E --> F[Métricas vs GT<br/>AbsRel, RMSE, delta...]
    F --> G[Tabla + CSV<br/>+ visualizaciones]
```

A lo largo del *notebook* las celdas de texto explican cada etapa con el detalle suficiente para que el reporte sea **autoexplicativo**.

---
## 0. Pipeline general

```mermaid
flowchart TD
    A["SCARED - split oficial: 550 frames, datasets 1-7"] --> B
    B["Imagen RGB 1280x1024 px"] --> C1 & C2 & C3 & C4

    subgraph ENHANCEMENT ["Image Enhancement"]
        C1["None - baseline"]
        C2["Retinex SSR - Rahman 2004"]
        C3["EndoLMSPEC - Endo4IE - Garcia-Vega 2022"]
        C4["IAT EndoViT - Endo4IE - Wang 2022"]
    end

    C1 & C2 & C3 & C4 --> D1 & D2 & D3 & D4 & D5 & D6

    subgraph DEPTH ["Modelos — pesos SCARED (Dr. Espinosa Loera)"]
        D1["Monodepth2 - ResNet18 - epoch 19"]
        D2["MonoViT - MPViT-Small - epoch 19"]
        D3["EndoSfMLearner - DispResNet18"]
        D4["AF-SfMLearner - ResNet18 + AppFlow"]
        D5["MonoIIT - MPViT + Lighting - winner"]
        D6["weights19-MonoViT - MPViT + Lighting"]
    end

    D1 & D2 & D3 & D4 & D5 & D6 --> E["Median Scaling a mm"]
    E --> F["AbsRel - SqRel - RMSE - RMSELog - delta - FPS"]

    style ENHANCEMENT fill:#fff8e1,stroke:#E0A800,stroke-width:2px
    style DEPTH fill:#e8f4fd,stroke:#2766CB,stroke-width:2px
```

**Diseño factorial**: 5 enhancements × 6 modelos × 550 frames = **16,500 evaluaciones**


In [4]:
%matplotlib inline
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "tifffile", "scikit-image", "fvcore"])

import torch, tifffile, cv2
import numpy as np
print(f"torch  : {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")


torch  : 2.11.0+cu128
CUDA   : True


---
## 1. Configuración de rutas

### Estructura esperada en Google Drive

```
MyDrive/proyecto_integrador/
├── scared_raw/
├── Endo-Depth-and-Motion/        ← código Monodepth2 / Endo-Depth
├── EndoLMSPEC/
├── EndoViT/
├── EndoSLAM/
├── MonoViT/                      ← código MonoViT (también para MonoIIT)
├── AF-SfMLearner/                ← código AF-SfMLearner
└── scared weights/               ← pesos SCARED (Dr. Espinosa Loera)
    ├── monodepth2_weights/weights_19/                ← encoder.pth + depth.pth
    ├── monovit_weights/weights_19/                   ← encoder.pth + depth.pth
    ├── endosfmlearner_weights/11-09-03_58/           ← dispnet_model_best.pth.tar
    ├── afmlearner_weights/Model_trained_end_to_end/  ← encoder.pth + depth.pth
    ├── monoIIT_weights/trained-winner-weights/       ← encoder.pth + depth.pth (+ lighting.pth)
    ├── weights_19_MonoViT/weights_19/                ← encoder.pth + depth.pth (+ lighting.pth)
    └── endodac_weights/  (pendiente — código propio)
```

> **Nota sobre MonoIIT**: el Dr. Espinosa indicó que los pesos de MonoIIT
> se ejecutan con el código de testing de MonoViT (misma arquitectura MPViT,
> con un módulo de iluminación adicional que no se usa en inferencia de profundidad).

### Tabla de pesos — Avance 5 vs Avance 4

| Modelo | Avance 4 (pesos originales) | Avance 5 (pesos SCARED) |
|---|---|---|
| Monodepth2 | Hamlyn | SCARED |
| MonoViT | KITTI | SCARED |
| EndoSfMLearner | EndoSLAM | SCARED |
| AF-SfMLearner | Endovis (paper) | SCARED |
| MonoIIT | — (no en A4) | SCARED |
| weights19-MonoViT | — (no en A4) | SCARED |

## 1. Configuración del entorno y el *split* oficial

Esta celda define todas las rutas (en **Google Colab** los datos viven en Drive; en **local** en disco `E:`) y carga el ***split* oficial**.

### ¿Qué es el *split* de prueba?

Un *split* es simplemente una **lista fija de qué fotogramas se usan para evaluar**, para que todos los trabajos midan sobre exactamente los mismos datos. El de AF-SfMLearner (`endovis/test_files.txt`) tiene 550 líneas con este formato:

```
dataset3/keyframe4 390 l
└─────┬─────┘     └┬┘ └┬┘
 carpeta        frame  lado (left)
```

Cada línea dice: *"el fotograma 390 del keyframe 4 del dataset 3, cámara izquierda"*. La función `load_split()` traduce eso a tuplas `(dataset_3, keyframe_4, 390)` que el resto del *notebook* consume.

> **Por qué importa:** usar este *split* es lo que hace que nuestros resultados sean **comparables** con AF-SfMLearner, MonoViT y los demás artículos — no es un detalle administrativo, es el corazón del rigor experimental.

In [ ]:
from pathlib import Path
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE          = Path("/content/drive/MyDrive/proyecto_integrador")
    SCARED_ROOT   = BASE / "scared_raw"
    EDAM_PATH     = BASE / "Endo-Depth-and-Motion"    # codigo Monodepth2
    LMSPEC_PATH   = BASE / "EndoLMSPEC"
    IAT_PATH      = BASE / "EndoViT"
    ENDOSLAM_PATH = BASE / "EndoSLAM"
    MONOVIT_PATH  = BASE / "MonoViT"
    AFSFM_PATH    = BASE / "AF-SfMLearner"
    STTN_PATH     = BASE / "Endo-STTN"

    W = BASE / "scared weights"
    W_MONO2   = W / "monodepth2_weights" / "weights_19"
    W_MONOVIT = W / "monovit_weights" / "weights_19"
    W_ENDOSFM = W / "endosfmlearner_weights" / "11-09-03_58"
    W_AFSFM   = W / "afmlearner_weights" / "Model_trained_end_to_end"
    W_MONOIIT = W / "monoIIT_weights" / "trained-winner-weights"
    W_W19MONO = W / "weights_19_MonoViT" / "weights_19"

    REPO_ROOT = Path("/content/repo_52")
    if not REPO_ROOT.exists():
        subprocess.check_call([
            "git", "clone", "--depth=1",
            "https://github.com/jmtoral/proyecto_integrador_52.git",
            str(REPO_ROOT)
        ])
    else:
        # repo ya clonado en una sesion previa: actualizar para traer data/splits, etc.
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"])
    subprocess.check_call(["git","-C",str(REPO_ROOT),"config","user.email","jmtoralcruz@gmail.com"])
    subprocess.check_call(["git","-C",str(REPO_ROOT),"config","user.name","jmtoral"])
    OUT_DIR    = REPO_ROOT / "outcomes" / "avance5_newversion"
    SPLIT_FILE = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"
    FRAMES_CACHE = Path("/content/split_frames")     # frames+GT extraidos (efimero, rapido)

else:
    BASE          = Path("E:/scared_wights_complete/scared weights")
    SCARED_ROOT   = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    EDAM_PATH     = Path("E:/Endo-Depth-and-Motion")
    LMSPEC_PATH   = Path("E:/EndoLMSPEC")
    IAT_PATH      = Path("E:/EndoVit")
    ENDOSLAM_PATH = Path("E:/EndoSLAM")
    MONOVIT_PATH  = Path("E:/MonoViT")
    AFSFM_PATH    = Path("E:/AF-SfMLearner")
    STTN_PATH     = Path("E:/Endo-STTN")

    W = BASE
    W_MONO2   = W / "monodepth2_weights" / "weights_19"
    W_MONOVIT = W / "monovit_weights" / "weights_19"
    W_ENDOSFM = W / "endosfmlearner_weights" / "11-09-03_58"
    W_AFSFM   = W / "afmlearner_weights" / "Model_trained_end_to_end"
    W_MONOIIT = W / "monoIIT_weights" / "trained-winner-weights"
    W_W19MONO = W / "weights_19_MonoViT" / "weights_19"

    REPO_ROOT  = Path(r"d:\Proyecto_Integrador\Corrreccion_Luz")
    OUT_DIR    = REPO_ROOT / "outcomes" / "avance5_newversion"
    SPLIT_FILE = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"
    FRAMES_CACHE = REPO_ROOT / "data" / "split_frames"

LMSPEC_WEIGHTS  = LMSPEC_PATH / "checkpoint" / "main_net" / "model_256_combined_SSIM5_1.pth"
IAT_WEIGHTS     = IAT_PATH / "Endo4IE" / "best_Epoch50_laplacian_histogan_loss.pth"
ENDOSFM_WEIGHTS = W_ENDOSFM / "dispnet_model_best.pth.tar"

# Pesos Endo-STTN (gen_00009.pth). En Colab se descargan de Drive si no existen.
STTN_CKPT_DIR    = STTN_PATH / "release_model" / "pretrained_model"
STTN_CKPT_NUMBER = "9"     # gen_00009.pth
STTN_WEIGHTS     = STTN_CKPT_DIR / "gen_00009.pth"
STTN_GDRIVE_ID   = "14sdaDejsxgRuzHBSuqH2xEpbxuqyWI-R"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_CACHE.mkdir(parents=True, exist_ok=True)
CAP_MM = 150.0

# ---- Split oficial AF-SfMLearner (endovis): 550 frames, datasets 1-7 ----
# Formato por linea:  "dataset3/keyframe4 390 l"
def load_split(split_file):
    items = []
    with open(split_file) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            folder, frame_id, _side = line.split()
            # folder = "dataset3/keyframe4" -> dataset_3, keyframe_4
            ds, kf = folder.split("/")
            ds_n = "dataset_" + ds.replace("dataset","")
            kf_n = "keyframe_" + kf.replace("keyframe","")
            items.append((ds_n, kf_n, int(frame_id)))
    return items

SPLIT_ITEMS = load_split(SPLIT_FILE)   # lista de (dataset_N, keyframe_M, frame_id)

print(f"Entorno : {'Colab' if IN_COLAB else 'Local'}")
print(f"OUT_DIR : {OUT_DIR}")
print(f"Split   : {len(SPLIT_ITEMS)} frames  (datasets {sorted({d for d,_,_ in SPLIT_ITEMS})})")
for n, p in [("SCARED_ROOT",SCARED_ROOT),("LMSPEC_WEIGHTS",LMSPEC_WEIGHTS),
             ("IAT_WEIGHTS",IAT_WEIGHTS),("ENDOSFM_WEIGHTS",ENDOSFM_WEIGHTS),
             ("W_MONO2",W_MONO2),("W_MONOVIT",W_MONOVIT),
             ("W_AFSFM",W_AFSFM),("W_MONOIIT",W_MONOIIT),
             ("W_W19MONO",W_W19MONO),("STTN_PATH",STTN_PATH)]:
    print(f"  {n:20s}: {'OK' if Path(p).exists() else 'NO ENCONTRADO'}")


---
### Extracción del split oficial (550 frames)

El split de AF-SfMLearner (`endovis/test_files.txt`) referencia **frames concretos dentro de los vídeos** de cada *keyframe* (`data/rgb.mp4`) y su *ground-truth* de profundidad (`data/scene_points.tar.gz`, un `.tiff` por frame). Esta celda extrae **solo** los frames listados en el split —imagen + GT, recortados a la mitad izquierda `[0:1024,:]` como en el código oficial— y los cachea en disco para no re-extraer.

Mapeo oficial: para el frame `N` → imagen = frame `N` del vídeo, GT = `scene_points{N-1:06d}.tiff`.

## 2. Extracción de los 550 fotogramas del *split*

Aquí está la diferencia técnica más grande respecto a versiones anteriores. El *split* referencia fotogramas que **no existen como archivos sueltos** dentro de los `.zip` de SCARED — están **dentro de dos contenedores comprimidos** por cada *keyframe*:

| Contenedor | Qué contiene |
|---|---|
| `data/rgb.mp4` | El **vídeo** completo del *keyframe* (~830 fotogramas) |
| `data/scene_points.tar.gz` | El ***ground-truth*** de profundidad, un `.tiff` por fotograma |

### Cómo se mapea cada fotograma a su *ground-truth*

El código sigue **exactamente** la convención oficial de AF-SfMLearner:

- **Imagen** del fotograma `N` → fotograma `N` del `rgb.mp4`
- **Profundidad** del fotograma `N` → archivo `scene_points{N-1:06d}.tiff` (¡ojo: índice **desfasado en 1**!)
- Ambos se **recortan a la mitad izquierda** `[0:1024, :]` (SCARED es estéreo; usamos la cámara izquierda)
- El `.tiff` de GT guarda coordenadas 3D; tomamos el **canal Z** (la profundidad en mm)

### Caché persistente (para no re-extraer nunca más)

La extracción es **lenta** (decodificar vídeo es trabajo de CPU). Por eso, la **primera vez** se extraen los 550 fotogramas y se guardan **empaquetados** en `BASE/split_frames.npz` (en tu Drive). En **todas las sesiones siguientes**, el *notebook* detecta ese `.npz` y **carga todo en segundos**, saltándose la extracción. Para forzar una re-extracción, define `FORCE_REEXTRACT = True` antes de esta celda.

In [ ]:
import io, zipfile, tarfile, cv2, numpy as np, tifffile
from collections import defaultdict
from tqdm import tqdm

# Caché persistente empaquetado en Drive (sobrevive entre sesiones de Colab)
NPZ_CACHE = BASE / "split_frames.npz"
def _key(ds, kf, fid): return f"{ds}|{kf}|{fid}"

def _frame_paths(ds, kf, fid):
    base = FRAMES_CACHE / ds / kf
    return base / f"img_{fid:010d}.png", base / f"gt_{fid:010d}.npy"

def extract_split_frames(split_items, scared_root, force=False):
    """Extrae imagen+GT de cada (ds,kf,frame). Scratch en FRAMES_CACHE (rapido)."""
    by_kf = defaultdict(list)
    for ds, kf, fid in split_items:
        by_kf[(ds, kf)].append(fid)
    n_done = 0
    for (ds, kf), fids in tqdm(by_kf.items(), desc="Keyframes del split"):
        fids = sorted(set(fids))
        pend = [f for f in fids if force or not _frame_paths(ds, kf, f)[0].exists()
                                        or not _frame_paths(ds, kf, f)[1].exists()]
        if not pend: continue
        (FRAMES_CACHE / ds / kf).mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(scared_root / f"{ds}.zip") as z:
            sp_bytes = z.read(f"{ds}/{kf}/data/scene_points.tar.gz")
            with tarfile.open(fileobj=io.BytesIO(sp_bytes)) as t:
                tnames = {n.split("/")[-1]: n for n in t.getnames() if n.endswith(".tiff")}
                gt_cache = {}
                for fid in pend:
                    key = f"scene_points{fid-1:06d}.tiff"
                    if key not in tnames: gt_cache[fid] = None; continue
                    raw = t.extractfile(tnames[key]).read()
                    tiff = tifffile.imread(io.BytesIO(raw))
                    dz = tiff[..., 2].astype(np.float32) if tiff.ndim == 3 else tiff.astype(np.float32)
                    dz = dz[0:1024, :]; dz[dz <= 0] = np.nan; gt_cache[fid] = dz
            tmp_mp4 = FRAMES_CACHE / "_tmp.mp4"
            with open(tmp_mp4, "wb") as fout: fout.write(z.read(f"{ds}/{kf}/data/rgb.mp4"))
            cap = cv2.VideoCapture(str(tmp_mp4)); want = set(pend); maxf = max(pend)
            idx = 0; got = {}
            while idx <= maxf:
                ok, frame = cap.read()
                if not ok: break
                if idx in want: got[idx] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)[0:1024, :]
                idx += 1
            cap.release(); tmp_mp4.unlink(missing_ok=True)
        for fid in pend:
            img_p, gt_p = _frame_paths(ds, kf, fid)
            if fid in got: cv2.imwrite(str(img_p), cv2.cvtColor(got[fid], cv2.COLOR_RGB2BGR))
            if gt_cache.get(fid) is not None: np.save(gt_p, gt_cache[fid])
            n_done += 1
    print(f"Extracción lista — {n_done} frames nuevos")

# Diccionario en memoria: clave -> (img_uint8, gt_float32|None)
SPLIT_DATA = {}

if NPZ_CACHE.exists() and not globals().get("FORCE_REEXTRACT", False):
    # ---- Carga rapida desde Drive (segundos) ----
    print(f"Cargando caché empaquetado: {NPZ_CACHE.name}")
    _npz = np.load(NPZ_CACHE, allow_pickle=True)
    for ds, kf, fid in SPLIT_ITEMS:
        k = _key(ds, kf, fid)
        ik, gk = "img_"+k, "gt_"+k
        if ik in _npz.files:
            gt = _npz[gk] if gk in _npz.files else None
            if gt is not None and gt.size == 1 and np.isnan(gt).all(): gt = None
            SPLIT_DATA[k] = (_npz[ik], gt)
    print(f"Caché cargado — {len(SPLIT_DATA)} frames listos (sin re-extraer)")
else:
    # ---- Primera vez: extraer y empaquetar en Drive ----
    extract_split_frames(SPLIT_ITEMS, SCARED_ROOT)
    save_dict = {}
    for ds, kf, fid in SPLIT_ITEMS:
        img_p, gt_p = _frame_paths(ds, kf, fid)
        if not img_p.exists(): continue
        k = _key(ds, kf, fid)
        img = cv2.cvtColor(cv2.imread(str(img_p), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        gt = np.load(gt_p) if gt_p.exists() else np.array([np.nan], np.float32)
        save_dict["img_"+k] = img
        save_dict["gt_"+k] = gt
        SPLIT_DATA[k] = (img, gt if gt.size > 1 else None)
    np.savez_compressed(NPZ_CACHE, **save_dict)
    print(f"Caché empaquetado guardado en Drive: {NPZ_CACHE}  ({len(SPLIT_DATA)} frames)")
    print("En sesiones futuras se cargara de aqui en segundos (no re-extrae).")


---
## 2. Cargar Modelos — Pesos SCARED

### 2a. Monodepth2 (ResNet-18, entrenado en SCARED)

In [6]:
import torch
import numpy as np

sys.path.insert(0, str(EDAM_PATH / "apps" / "depth_estimate"))
from resnet_encoder import ResnetEncoder
from depth_decoder import DepthDecoder

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

mono2_encoder = ResnetEncoder(18, False)
enc_dict = torch.load(W_MONO2 / "encoder.pth", map_location=DEVICE)
MONO2_H = enc_dict.get("height", 192)
MONO2_W = enc_dict.get("width",  640)
filtered = {k: v for k, v in enc_dict.items() if k in mono2_encoder.state_dict()}
mono2_encoder.load_state_dict(filtered)
mono2_encoder.to(DEVICE).eval()

mono2_decoder = DepthDecoder(num_ch_enc=mono2_encoder.num_ch_enc, scales=range(4))
mono2_decoder.load_state_dict(torch.load(W_MONO2 / "depth.pth", map_location=DEVICE))
mono2_decoder.to(DEVICE).eval()

print(f"Monodepth2 (SCARED): {MONO2_H}x{MONO2_W} — cargado OK")


Dispositivo: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Monodepth2 (SCARED): 256x320 — cargado OK


### 2b. MonoViT (MPViT-Small, entrenado en SCARED)

In [ ]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm", "einops"])

import importlib.util, types
import torch.nn as nn
from collections import OrderedDict

# Los pesos del Dr. Espinosa usan encoder MPViT (MonoViT) pero DOS decoders distintos
# segun el modelo:
#   - MonoViT  -> DepthDecoder simple Monodepth2 (num_ch_dec = num_ch_enc)
#   - MonoIIT  -> HR-Depth (el del paper MonoViT, con convs.f4 / X_00 / attention)
# load_monovit_pair detecta automaticamente cual usar inspeccionando las keys.

_NETDIR = MONOVIT_PATH / "networks"

# --- 1. cargar submodulos del repo MonoViT aislados ---
for _k in list(sys.modules):
    if _k == "networks" or _k.startswith("networks."):
        del sys.modules[_k]
_pkg = types.ModuleType("networks"); _pkg.__path__ = [str(_NETDIR)]
sys.modules["networks"] = _pkg
def _load_sub(name, fname):
    spec = importlib.util.spec_from_file_location(f"networks.{name}", str(_NETDIR / fname))
    m = importlib.util.module_from_spec(spec); sys.modules[f"networks.{name}"] = m
    spec.loader.exec_module(m); setattr(_pkg, name, m); return m

_load_sub("hr_layers", "hr_layers.py")
_hr_dec     = _load_sub("hr_decoder", "hr_decoder.py")
mpvit_small = _load_sub("mpvit", "mpvit.py").mpvit_small
DepthDecoderHR = _hr_dec.DepthDecoder

# --- 2. DepthDecoder simple Monodepth2 (num_ch_dec = num_ch_enc) ---
class _Conv3x3(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.pad = nn.ReflectionPad2d(1)
        self.conv = nn.Conv2d(int(i), int(o), 3)
    def forward(self, x): return self.conv(self.pad(x))
class _ConvBlock(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.conv = _Conv3x3(i, o); self.nonlin = nn.ELU(inplace=True)
    def forward(self, x): return self.nonlin(self.conv(x))
def _up(x): return nn.functional.interpolate(x, scale_factor=2, mode="nearest")

class MonoViTDepthDecoderSimple(nn.Module):
    def __init__(self, num_ch_enc=[64,128,216,288,288], scales=range(4), use_skips=True):
        super().__init__()
        self.scales=list(scales); self.use_skips=use_skips
        self.num_ch_enc=np.array(num_ch_enc); self.num_ch_dec=np.array(num_ch_enc)
        self.convs=OrderedDict()
        for i in range(4,-1,-1):
            ci=self.num_ch_enc[-1] if i==4 else self.num_ch_dec[i+1]
            self.convs[("upconv",i,0)]=_ConvBlock(ci,self.num_ch_dec[i])
            ci=self.num_ch_dec[i]
            if self.use_skips and i>0: ci+=self.num_ch_enc[i-1]
            self.convs[("upconv",i,1)]=_ConvBlock(ci,self.num_ch_dec[i])
        for s in self.scales:
            self.convs[("dispconv",s)]=_Conv3x3(self.num_ch_dec[s],1)
        self.decoder=nn.ModuleList(list(self.convs.values())); self.sigmoid=nn.Sigmoid()
    def forward(self, feats):
        out={}; x=feats[-1]
        for i in range(4,-1,-1):
            x=self.convs[("upconv",i,0)](x); x=[_up(x)]
            if self.use_skips and i>0: x+=[feats[i-1]]
            x=torch.cat(x,1); x=self.convs[("upconv",i,1)](x)
            if i in self.scales: out[("disp",i)]=self.sigmoid(self.convs[("dispconv",i)](x))
        return out

def _build_decoder_for(weights_dir, device):
    """Detecta el tipo de decoder segun las keys del checkpoint."""
    sd = torch.load(weights_dir / "depth.pth", map_location=device)
    is_hr = any(k.startswith("convs.f") or "X_00" in k or "_attention" in k for k in sd)
    if is_hr:
        dec = DepthDecoderHR()
        kind = "HR-Depth"
    else:
        dec = MonoViTDepthDecoderSimple([64,128,216,288,288], scales=range(4))
        kind = "Monodepth2-simple"
    dec.load_state_dict(sd)
    return dec, kind

def load_monovit_pair(weights_dir, device):
    encoder = mpvit_small()
    encoder.num_ch_enc = [64, 128, 216, 288, 288]
    enc_dict = torch.load(weights_dir / "encoder.pth", map_location=device)
    h = enc_dict.get("height", 192); w = enc_dict.get("width", 640)
    encoder.load_state_dict({k: v for k, v in enc_dict.items()
                             if k in encoder.state_dict()})
    encoder.to(device).eval()
    decoder, kind = _build_decoder_for(weights_dir, device)
    decoder.to(device).eval()
    print(f"  decoder: {kind}")
    return encoder, decoder, h, w

monovit_enc, monovit_dec, MONOVIT_H, MONOVIT_W = load_monovit_pair(W_MONOVIT, DEVICE)
print(f"MonoViT (SCARED): {MONOVIT_H}x{MONOVIT_W} — cargado OK")

### 2c. EndoSfMLearner (DispResNet-18, entrenado en SCARED)

In [ ]:
import importlib.util

_endosfm_dir = ENDOSLAM_PATH / "EndoSfMLearner"
if str(_endosfm_dir) not in sys.path:
    sys.path.insert(0, str(_endosfm_dir))

_spec = importlib.util.spec_from_file_location(
    "_endosfm_models_a5",
    str(_endosfm_dir / "models" / "__init__.py"),
    submodule_search_locations=[str(_endosfm_dir / "models")]
)
endosfm_models_a5 = importlib.util.module_from_spec(_spec)
sys.modules["_endosfm_models_a5"] = endosfm_models_a5
_spec.loader.exec_module(endosfm_models_a5)

endosfm_scared = endosfm_models_a5.DispResNet(18, False).to(DEVICE)
w = torch.load(ENDOSFM_WEIGHTS, map_location=DEVICE)
endosfm_scared.load_state_dict(w["state_dict"])
endosfm_scared.eval()

n = sum(p.numel() for p in endosfm_scared.parameters())
print(f"EndoSfMLearner (SCARED): {n/1e6:.2f} M — cargado OK")


### 2d. AF-SfMLearner (ResNet-18 + Appearance Flow, entrenado en SCARED)

In [ ]:
# AF-SfMLearner usa la misma arquitectura que Endo-Depth (ResnetEncoder + DepthDecoder)
afsfm_scared_enc = ResnetEncoder(18, False)
enc_w = torch.load(W_AFSFM / "encoder.pth", map_location=DEVICE)
AFSFM_H = enc_w.get("height", 256)
AFSFM_W = enc_w.get("width",  320)
filtered = {k: v for k, v in enc_w.items() if k in afsfm_scared_enc.state_dict()}
afsfm_scared_enc.load_state_dict(filtered)
afsfm_scared_enc.to(DEVICE).eval()

afsfm_scared_dec = DepthDecoder(num_ch_enc=afsfm_scared_enc.num_ch_enc, scales=range(4))
afsfm_scared_dec.load_state_dict(torch.load(W_AFSFM / "depth.pth", map_location=DEVICE))
afsfm_scared_dec.to(DEVICE).eval()

print(f"AF-SfMLearner (SCARED): {AFSFM_H}x{AFSFM_W} — cargado OK")


### 2e. MonoIIT (MPViT + Lighting, entrenado en SCARED)

**Nota del Dr. Espinosa Loera**: los pesos de MonoIIT se ejecutan con el código de MonoViT.
La arquitectura base es MPViT-Small (igual que MonoViT). El módulo de iluminación (`lighting.pth`)
se usa durante el entrenamiento; para inferencia de profundidad se usan solo `encoder.pth` + `depth.pth`.


In [ ]:
# MonoIIT: misma arquitectura/código que MonoViT, pesos diferentes
monoIIT_enc, monoIIT_dec, MONOIIT_H, MONOIIT_W = load_monovit_pair(W_MONOIIT, DEVICE)
print(f"MonoIIT (SCARED): {MONOIIT_H}x{MONOIIT_W} — cargado OK")
if (W_MONOIIT / "lighting.pth").exists():
    print("  lighting.pth presente (no usado en inferencia de profundidad)")

### 2f. weights19-MonoViT 

Esta carpeta tiene la misma estructura que MonoIIT (con `lighting.pth`).
Podría ser MonoIIF, el tercer modelo propuesto. **Pendiente de confirmación.**


In [ ]:
# weights_19_MonoViT: misma estructura, posible MonoIIF
w19_enc, w19_dec, W19_H, W19_W = load_monovit_pair(W_W19MONO, DEVICE)
print(f"weights19-MonoViT (SCARED): {W19_H}x{W19_W} — cargado OK")
if (W_W19MONO / "lighting.pth").exists():
    print("  lighting.pth presente — probable MonoIIF")

---
## 3. Métodos de Image Enhancement

Idénticos al Avance 4 — mismas funciones de corrección.

## 4. Métodos de realce de imagen (*enhancement*)

El experimento compara **5 estrategias** de pre-procesamiento aplicadas a la imagen *antes* de estimar profundidad. La hipótesis es que corregir la iluminación/reflejos podría ayudar al modelo de profundidad.

| Método | Tipo | Idea |
|---|---|---|
| **none** | — | Imagen original, sin tocar (línea base) |
| **retinex** | Clásico | *Single-Scale Retinex*: separa reflectancia de iluminación para homogenizar el brillo |
| **endolmspec** | *Deep* | EndoLMSPEC: pirámide Laplaciana + U-Nets, entrenado para realce endoscópico |
| **iat** | *Deep* | IAT (*Illumination-Adaptive Transformer*): corrige exposición con mapeo local + global |
| **endosttn** | *Deep · temporal* | **Endo-STTN** (nuevo): elimina reflejos especulares |

### Endo-STTN — el método nuevo, y por qué es especial

A diferencia de los otros cuatro (que procesan **una imagen aislada**), Endo-STTN es **temporal**: trata la secuencia de fotogramas de un *keyframe* como un vídeo y rellena (*inpainting*) las zonas de **reflejo especular** usando lo que se ve en los **fotogramas vecinos**. El procedimiento:

1. Detecta los reflejos especulares (zonas muy brillantes) → genera una **máscara**.
2. Un *transformer* espacio-temporal "mira" una ventana de **10 fotogramas vecinos** y reconstruye el contenido oculto bajo el reflejo.
3. Devuelve cada fotograma con los reflejos sustituidos por tejido plausible.

> Por eso el *split* de 550 fotogramas **consecutivos** es ideal para Endo-STTN: necesita vecindad temporal, algo imposible con los *keyframes* sueltos de antes. La salida se **cachea por *keyframe*** para no re-procesar la misma secuencia.

In [ ]:
import cv2, numpy as np, torch, torchvision.transforms as T
import importlib.util, types, subprocess

# Dependencias de EndoLMSPEC/EndoViT
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "IQA_pytorch", "path"])

# EndoLMSPEC
def _load_endolmspec(lmspec_path, device):
    _orig = sys.path.copy()
    # Excluir EndoSLAM, HADepth y EndoViT — todos tienen 'utils' que colisiona con utils.pyramids
    clean = [str(lmspec_path)] + [
        p for p in sys.path
        if "EndoSLAM" not in p and "endosfm" not in p.lower()
        and "HADepth" not in p and "EndoViT" not in p and "EndoVit" not in p]
    for k in list(sys.modules):
        if k in ("utils","generator","unet") or k.startswith("utils."): del sys.modules[k]
    try:
        sys.path = clean
        spec = importlib.util.spec_from_file_location("generator", lmspec_path/"generator.py")
        mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
        Generator = mod.Generator
    finally:
        sys.path = _orig
    return Generator(n_channels=3, device=device, bilinear=False), Generator

lmspec_net, _ = _load_endolmspec(LMSPEC_PATH, DEVICE)
lmspec_net.load_state_dict(torch.load(LMSPEC_WEIGHTS, map_location=DEVICE))
lmspec_net.to(DEVICE).eval()
print("EndoLMSPEC OK")

# IAT — limpiar utils de EndoLMSPEC antes para que IAT cargue su propio utils
for k in list(sys.modules):
    if k == "utils" or k.startswith("utils."): del sys.modules[k]
sys.modules["imp"] = types.ModuleType("imp")
_iat_model_path = IAT_PATH / "experiments" / "model" / "IAT_main.py"
_spec = importlib.util.spec_from_file_location("IAT_main_a5", _iat_model_path)
_iat_mod = importlib.util.module_from_spec(_spec)
_iat_path = str(IAT_PATH / "experiments")
if _iat_path not in sys.path: sys.path.insert(0, _iat_path)
_spec.loader.exec_module(_iat_mod)
iat_net = _iat_mod.IAT(in_dim=3, with_global=True, type="exp")
iat_net.load_state_dict(torch.load(IAT_WEIGHTS, map_location=DEVICE))
iat_net.to(DEVICE).eval()
print("IAT OK")

def correct_none(img): return img

def correct_retinex(img, sigma=30):
    img_f = img.astype(np.float32) + 1.0
    result = np.zeros_like(img_f)
    for c in range(3):
        blur = cv2.GaussianBlur(img_f[:,:,c],(0,0),sigma)
        result[:,:,c] = np.log(img_f[:,:,c]) - np.log(blur+1.0)
    result -= result.min()
    return (result/(result.max()+1e-8)*255).astype(np.uint8)

def correct_endolmspec(img):
    t = T.ToTensor()(img).to(DEVICE)
    with torch.no_grad(): _, out = lmspec_net(t)
    return (out["subnet_16"][0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)

def correct_iat(img):
    t = torch.from_numpy(img.astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): _, _, enh = iat_net(t)
    return (enh[0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)

CORRECTIONS = {"none":correct_none,"retinex":correct_retinex,
               "endolmspec":correct_endolmspec,"iat":correct_iat}
print(f"Enhancements: {list(CORRECTIONS.keys())}")
# ---------------------------------------------------------------------
# Endo-STTN (Spatio-Temporal Transformer Network) — quita reflejos
# especulares usando informacion temporal de frames vecinos.
# Es TEMPORAL: procesa la secuencia completa de un keyframe de una vez.
# ---------------------------------------------------------------------
import importlib.util as _ilu

# Descargar pesos en Colab si no existen
if IN_COLAB and not STTN_WEIGHTS.exists():
    STTN_CKPT_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([sys.executable,"-m","pip","install","-q","gdown"])
    import gdown
    gdown.download(id=STTN_GDRIVE_ID, output=str(STTN_WEIGHTS), quiet=False)

def _load_endo_sttn(sttn_path, ckpt, device):
    _orig_path = sys.path.copy()
    _orig_mods = {k: sys.modules[k] for k in list(sys.modules)
                  if k == "core" or k.startswith("core.") or k == "model" or k.startswith("model.")}
    for k in list(sys.modules):
        if k == "core" or k.startswith("core.") or k == "model" or k.startswith("model."):
            del sys.modules[k]
    try:
        sys.path.insert(0, str(sttn_path))
        net_mod = __import__("model.sttn", fromlist=["InpaintGenerator"])
        utils_mod = __import__("core.utils", fromlist=["Stack","ToTorchFormatTensor"])
        model = net_mod.InpaintGenerator().to(device)
        data = torch.load(ckpt, map_location=device)
        model.load_state_dict(data["netG"])
        model.eval()
        Stack = utils_mod.Stack; ToTorch = utils_mod.ToTorchFormatTensor
    finally:
        sys.path = _orig_path
        for k in list(sys.modules):
            if k == "core" or k.startswith("core.") or k == "model" or k.startswith("model."):
                del sys.modules[k]
        sys.modules.update(_orig_mods)
    return model, Stack, ToTorch

STTN_W, STTN_H = 288, 288       # tamaño de entrada del modelo (config oficial)
STTN_REF_LEN, STTN_STRIDE = 10, 5
_sttn_ok = STTN_WEIGHTS.exists()
if _sttn_ok:
    sttn_model, _Stack, _ToTorch = _load_endo_sttn(STTN_PATH, STTN_WEIGHTS, DEVICE)
    from torchvision import transforms as _tvt
    _sttn_to_tensors = _tvt.Compose([_Stack(), _ToTorch()])
    print("Endo-STTN OK")
else:
    print("Endo-STTN: pesos no encontrados, se omitira (subir gen_00009.pth a Drive)")

def _sttn_specular_mask(img_rgb, dil=8):
    """Mascara binaria de reflejos especulares (zonas muy brillantes)."""
    L = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)[:,:,0].astype(np.float32)
    m = (L >= np.percentile(L, 97)).astype(np.uint8)
    if dil:
        m = cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(dil,dil)), iterations=1)
    return m

def _sttn_get_ref_index(neighbor_ids, length):
    return [i for i in range(0, length, STTN_REF_LEN) if i not in neighbor_ids]

@torch.no_grad()
def endo_sttn_inpaint_sequence(frames_rgb):
    """frames_rgb: lista de np.uint8 HxWx3. Devuelve lista inpainted (mismo tamano original)."""
    if not _sttn_ok:
        return frames_rgb
    H0, W0 = frames_rgb[0].shape[:2]
    pil_frames = [pil.fromarray(f).resize((STTN_W, STTN_H), pil.LANCZOS) for f in frames_rgb]
    masks_np   = [cv2.resize(_sttn_specular_mask(f),(STTN_W,STTN_H),interpolation=cv2.INTER_NEAREST)
                  for f in frames_rgb]
    pil_masks  = [pil.fromarray((m*255).astype(np.uint8)) for m in masks_np]
    vlen = len(pil_frames)

    feats = _sttn_to_tensors(pil_frames).unsqueeze(0)*2-1
    masks = _sttn_to_tensors(pil_masks).unsqueeze(0)
    feats, masks = feats.to(DEVICE), masks.to(DEVICE)
    bin_masks = [np.expand_dims((np.array(m)!=0).astype(np.uint8),2) for m in pil_masks]
    raw_frames = [np.array(f).astype(np.uint8) for f in pil_frames]

    feats = sttn_model.encoder((feats*(1-masks).float()).view(vlen,3,STTN_H,STTN_W))
    _, c, fh, fw = feats.size()
    feats = feats.view(1, vlen, c, fh, fw)

    comp = [None]*vlen
    for f in range(0, vlen, STTN_STRIDE):
        nb_ids = [i for i in range(max(0,f-STTN_STRIDE), min(vlen,f+STTN_STRIDE+1))]
        ref_ids = _sttn_get_ref_index(nb_ids, vlen)
        pred_feat = sttn_model.infer(feats[0, nb_ids+ref_ids], masks[0, nb_ids+ref_ids])
        pred_img = torch.tanh(sttn_model.decoder(pred_feat[:len(nb_ids)]))
        pred_img = ((pred_img+1)/2).cpu().permute(0,2,3,1).numpy()*255
        for i, idx in enumerate(nb_ids):
            # solo rellenar dentro de la mascara; fuera, mantener el original
            img = pred_img[i].astype(np.uint8)*bin_masks[idx] + raw_frames[idx]*(1-bin_masks[idx])
            comp[idx] = img if comp[idx] is None else (comp[idx]*0.5 + img*0.5).astype(np.uint8)
    # volver al tamano original
    return [cv2.resize(c, (W0, H0), interpolation=cv2.INTER_LANCZOS4) for c in comp]

# Cache por keyframe para no re-inpaint la misma secuencia
_STTN_CACHE = {}
def correct_endo_sttn(img_rgb, ds=None, kf=None, fid=None):
    """Si se pasa (ds,kf,fid), usa inpainting temporal de toda la secuencia del keyframe.
    Si no, hace fallback espacial (1 solo frame)."""
    if not _sttn_ok:
        return img_rgb
    if ds is None:
        return endo_sttn_inpaint_sequence([img_rgb])[0]
    key = (ds, kf)
    if key not in _STTN_CACHE:
        fids = SPLIT_BY_KF[key]
        seq = [load_split_frame(ds, kf, f)[0] for f in fids]
        out = endo_sttn_inpaint_sequence(seq)
        _STTN_CACHE.clear()  # mantener memoria baja: solo 1 keyframe a la vez
        _STTN_CACHE[key] = {f: o for f, o in zip(fids, out)}
    return _STTN_CACHE[key][fid]

CORRECTIONS = {"none":correct_none,"retinex":correct_retinex,
               "endolmspec":correct_endolmspec,"iat":correct_iat,
               "endosttn":correct_endo_sttn}
print(f"Enhancements: {list(CORRECTIONS.keys())}")


---
## 4. Funciones de inferencia y métricas

## 3. Modelos de profundidad y métricas de evaluación

### Los 6 modelos (todos re-entrenados en SCARED por el Dr. Espinosa Loera)

| Modelo | Arquitectura |
|---|---|
| **Monodepth2** | ResNet-18 + *DepthDecoder* (auto-supervisado clásico) |
| **MonoViT** | *Encoder* MPViT-Small + decodificador estilo Monodepth2 |
| **EndoSfMLearner** | DispResNet-18 (especializado en endoscopía) |
| **AF-SfMLearner** | ResNet-18 + *Appearance Flow* (corrige cambios de brillo entre cuadros) |
| **MonoIIT** | MPViT + módulo de iluminación (mismo código que MonoViT, pesos distintos) |
| **w19-MonoViT** | Variante MonoViT (posible MonoIIF) |

Todos predicen **profundidad relativa**; para comparar con el GT en mm aplicamos ***median scaling*** (escalado por la mediana), el estándar en profundidad monocular auto-supervisada.

### Las métricas

Sobre los píxeles válidos del GT (profundidad en `(0, 150] mm`) calculamos:

| Métrica | Qué mide | Mejor |
|---|---|---|
| **AbsRel** | Error relativo absoluto medio `\|pred−gt\|/gt` | ↓ menor |
| **SqRel** | Error relativo cuadrático | ↓ menor |
| **RMSE** | Raíz del error cuadrático medio (mm) | ↓ menor |
| **RMSELog** | RMSE en escala logarítmica | ↓ menor |
| **δ < 1.25ᵏ** | Fracción de píxeles "casi correctos" (k = 1,2,3) | ↑ mayor |
| **Chamfer** | Distancia entre nubes de puntos 3D | ↓ menor |
| **AbsRel_spec / _nospec** | AbsRel **dentro** vs. **fuera** de zonas de reflejo | ↓ menor |

> La separación **spec / nospec** es clave para esta versión: nos permite medir si Endo-STTN (que ataca los especulares) realmente **mejora la profundidad justo en las zonas de reflejo**.

In [ ]:
import time, torch.nn.functional as F, PIL.Image as pil
from torchvision import transforms
from scipy.spatial import cKDTree
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.transform import resize as imresize

FX, FY, CX, CY = 1078.0, 1078.0, 640.0, 512.0

def _predict_monodepth2_style(img_rgb, enc, dec, h, w, device):
    """Monodepth2/AF-SfMLearner: ToTensor sin normalizacion ImageNet."""
    H, W = img_rgb.shape[:2]
    t = transforms.ToTensor()(pil.fromarray(img_rgb).resize((w,h),pil.LANCZOS)).unsqueeze(0).to(device)
    if device.type=="cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = dec(enc(t))
    if device.type=="cuda": torch.cuda.synchronize()
    ms = (time.perf_counter()-t0)*1000
    disp = F.interpolate(out[("disp",0)],(H,W),mode="bilinear",align_corners=False).squeeze().cpu().numpy()
    return 1.0/(1/100+(1/0.1-1/100)*disp), ms

def _predict_monovit_style(img_rgb, enc, dec, h, w, device):
    """MonoViT/MonoIIT: encoder mpvit_small + DepthDecoder separados.
    ToTensor sin normalizacion ImageNet (igual que evaluate_depth.py oficial)."""
    H, W = img_rgb.shape[:2]
    t = transforms.ToTensor()(pil.fromarray(img_rgb).resize((w,h),pil.LANCZOS)).unsqueeze(0).to(device)
    if device.type=="cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = dec(enc(t))
    if device.type=="cuda": torch.cuda.synchronize()
    ms = (time.perf_counter()-t0)*1000
    disp = F.interpolate(out[("disp",0)],(H,W),mode="bilinear",align_corners=False).squeeze().cpu().numpy()
    return 1.0/(1/100+(1/0.1-1/100)*disp), ms

def predict_mono2(img):     return _predict_monodepth2_style(img, mono2_encoder, mono2_decoder, MONO2_H, MONO2_W, DEVICE)
def predict_monovit(img):   return _predict_monovit_style(img, monovit_enc, monovit_dec, MONOVIT_H, MONOVIT_W, DEVICE)
def predict_afsfm(img):     return _predict_monodepth2_style(img, afsfm_scared_enc, afsfm_scared_dec, AFSFM_H, AFSFM_W, DEVICE)
def predict_monoIIT(img):   return _predict_monovit_style(img, monoIIT_enc, monoIIT_dec, MONOIIT_H, MONOIIT_W, DEVICE)
def predict_w19mono(img):   return _predict_monovit_style(img, w19_enc, w19_dec, W19_H, W19_W, DEVICE)

def predict_endosfm(img):
    H, W = img.shape[:2]
    r = imresize(img,(256,832)).astype(np.float32)
    t = torch.from_numpy(((r/255-0.45)/0.225).transpose(2,0,1)).unsqueeze(0).to(DEVICE)
    if DEVICE.type=="cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad(): d = endosfm_scared(t)
    if DEVICE.type=="cuda": torch.cuda.synchronize()
    ms = (time.perf_counter()-t0)*1000
    return 1/(imresize(d.squeeze().cpu().numpy(),(H,W))+1e-6), ms

DEPTH_MODELS = {
    "Monodepth2":      predict_mono2,
    "MonoViT":         predict_monovit,
    "EndoSfMLearner":  predict_endosfm,
    "AF-SfMLearner":   predict_afsfm,
    "MonoIIT":         predict_monoIIT,
    "w19-MonoViT":     predict_w19mono,
}

def specular_mask(img, pct=97, dil=15):
    L = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)[:,:,0].astype(np.float32)
    m = (L>=np.percentile(L,pct)).astype(np.uint8)
    return cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(dil,dil))).astype(bool)

def depth_to_pc(d, mask, fx=FX, fy=FY, cx=CX, cy=CY):
    H,W = d.shape; uu,vv = np.meshgrid(np.arange(W),np.arange(H))
    Z = d[mask]
    return np.stack([(uu[mask]-cx)*Z/fx,(vv[mask]-cy)*Z/fy,Z],axis=1)

def chamfer(p1,p2,n=50_000):
    if not len(p1) or not len(p2): return np.nan
    rng = np.random.default_rng(42)
    if len(p1)>n: p1=p1[rng.choice(len(p1),n,replace=False)]
    if len(p2)>n: p2=p2[rng.choice(len(p2),n,replace=False)]
    d1,_ = cKDTree(p2).query(p1); d2,_ = cKDTree(p1).query(p2)
    return float((d1.mean()+d2.mean())/2)

# Chamfer (KD-Trees en CPU) es el cuello de botella del experimento y NO esta
# en la tabla de métricas estándar reportada. Desactivado por defecto: baja ~7h a <1h.
# Ponlo en True solo si necesitas la metrica de nube de puntos 3D.
COMPUTE_CHAMFER = False

def compute_metrics(img_orig, img_corr, depth_rel, gt_mm, cap=150.0):
    valid = (~np.isnan(gt_mm))&(gt_mm>0)&(gt_mm<cap)
    if valid.sum()==0:
        return {k:np.nan for k in ["AbsRel","SqRel","RMSE","RMSELog",
                                   "delta_1","delta_2","delta_3",
                                   "Chamfer","AbsRel_spec","AbsRel_nospec","scale"]}
    scale = np.median(gt_mm[valid])/(np.median(depth_rel[valid])+1e-8)
    pred  = depth_rel*scale; d,gt = pred[valid],gt_mm[valid]
    spec  = specular_mask(img_orig); vs,vn = valid&spec, valid&~spec
    ratio = np.maximum(d/(gt+1e-8), gt/(d+1e-8))
    return {
        "scale":    round(float(scale),4),
        "AbsRel":   round(float(np.mean(np.abs(d-gt)/(gt+1e-8))),4),
        "SqRel":    round(float(np.mean((d-gt)**2/(gt+1e-8))),4),
        "RMSE":     round(float(np.sqrt(np.mean((d-gt)**2))),3),
        "RMSELog":  round(float(np.sqrt(np.mean((np.log(np.clip(d,1e-3,None))-np.log(np.clip(gt,1e-3,None)))**2))),4),
        "delta_1":  round(float(np.mean(ratio<1.25)),4),
        "delta_2":  round(float(np.mean(ratio<1.25**2)),4),
        "delta_3":  round(float(np.mean(ratio<1.25**3)),4),
        "Chamfer":  (round(chamfer(depth_to_pc(np.where(valid,pred,np.nan),valid),
                                   depth_to_pc(np.where(valid,gt_mm,np.nan),valid)),3)
                     if COMPUTE_CHAMFER else np.nan),
        "AbsRel_spec":  round(float(np.mean(np.abs(pred[vs]-gt_mm[vs])/(gt_mm[vs]+1e-8))),4) if vs.sum()>0 else np.nan,
        "AbsRel_nospec":round(float(np.mean(np.abs(pred[vn]-gt_mm[vn])/(gt_mm[vn]+1e-8))),4) if vn.sum()>0 else np.nan,
    }

print(f"Modelos  : {list(DEPTH_MODELS.keys())}")
print(f"Enhanc.  : {list(CORRECTIONS.keys())}")
print(f"Total    : {len(DEPTH_MODELS)*len(CORRECTIONS)*len(SPLIT_ITEMS)} evaluaciones")

---
## 5. Carga de datos SCARED

Protocolo: **split oficial de AF-SfMLearner** (`splits/endovis/test_files.txt`), 550 frames de los datasets 1–7.

In [ ]:
import numpy as np, cv2

def load_split_frame(ds, kf, fid):
    """Carga imagen RGB (uint8) y GT (mm, nan en invalidos) del caché en memoria."""
    img, gt = SPLIT_DATA[_key(ds, kf, fid)]
    return img, gt

# Frames consecutivos del mismo keyframe (para Endo-STTN, que es temporal)
from collections import defaultdict
SPLIT_BY_KF = defaultdict(list)
for ds, kf, fid in SPLIT_ITEMS:
    if _key(ds, kf, fid) in SPLIT_DATA:
        SPLIT_BY_KF[(ds, kf)].append(fid)
for k in SPLIT_BY_KF: SPLIT_BY_KF[k] = sorted(SPLIT_BY_KF[k])

_ds0, _kf0, _fid0 = SPLIT_ITEMS[0]
_img, _gt = load_split_frame(_ds0, _kf0, _fid0)
print(f"Ejemplo {_ds0}/{_kf0} frame {_fid0}: img {_img.shape}  "
      f"GT valido {(~np.isnan(_gt)).mean()*100:.1f}%" if _gt is not None else "sin GT")


---
## 6. Experimento factorial: 5 enhancements × 6 modelos × 550 frames (split oficial)

Total: **240 evaluaciones**

## 5. Bucle de evaluación con *checkpointing*

Se recorren los **550 fotogramas × 5 *enhancements* × 6 modelos**. Para cada combinación se mide el error de profundidad y los tiempos de cómputo.

Como es un experimento largo, el bucle **guarda el CSV cada 50 evaluaciones** y, si se reinicia, **reanuda donde quedó** (lee lo ya hecho y solo calcula lo que falta). Así, aunque la sesión de Colab se interrumpa, no se pierde el progreso.

> El *enhancement* `endosttn` se invoca de forma distinta (necesita saber `dataset/keyframe/frame` para usar su contexto temporal); el resto recibe solo la imagen.

In [ ]:
import pandas as pd, os
from tqdm import tqdm

assert len(DEPTH_MODELS)==6, f"Se esperan 6 modelos, hay {len(DEPTH_MODELS)}"
CKPT_CSV = OUT_DIR / "avance5_newversion_results.csv"
CKPT_EVERY = 50

# Reanudar si ya hay checkpoint
if CKPT_CSV.exists():
    df_prev = pd.read_csv(CKPT_CSV)
    done = set(zip(df_prev["Dataset"], df_prev["Keyframe"], df_prev["Frame"], df_prev["Método"], df_prev["Modelo"]))
    results = df_prev.to_dict("records")
    print(f"Reanudando: {len(done)} evaluaciones ya hechas")
else:
    done, results = set(), []

def _apply_correction(name, fn, img_rgb, ds, kf, fid):
    if name == "endosttn":
        return fn(img_rgb, ds=ds, kf=kf, fid=fid)
    return fn(img_rgb)

# Guard: si ya estan todas las evaluaciones, no recorrer nada (evita re-aplicar enhancements en vano)
_n_esperado = len(SPLIT_ITEMS) * len(CORRECTIONS) * 6
if len(done) >= _n_esperado:
    print(f"EXPERIMENTO COMPLETO: {len(done)} evaluaciones ya en {CKPT_CSV.name}. Nada que recalcular.")
    df = pd.DataFrame(results)
else:
    n_new = 0
    for (ds, kf, fid) in tqdm(SPLIT_ITEMS, desc="Frames del split"):
        # saltar el frame entero si TODAS sus combinaciones ya estan hechas (no re-aplica enhancements)
        if all((ds, kf, fid, cn, mn) in done for cn in CORRECTIONS for mn in DEPTH_MODELS):
            continue
        img_rgb, gt_mm = load_split_frame(ds, kf, fid)
        if gt_mm is None:
            continue
        for corr_name, corr_fn in CORRECTIONS.items():
            t0 = time.perf_counter()
            img_corr = _apply_correction(corr_name, corr_fn, img_rgb, ds, kf, fid)
            t_enh = (time.perf_counter()-t0)*1000
            for model_name, depth_fn in DEPTH_MODELS.items():
                if (ds, kf, fid, corr_name, model_name) in done:
                    continue
                depth_rel, t_inf = depth_fn(img_corr)
                m = compute_metrics(img_rgb, img_corr, depth_rel, gt_mm, CAP_MM)
                results.append({
                    "Dataset":ds,"Keyframe":kf,"Frame":fid,
                    "Modelo":model_name,"Método":corr_name,
                    "T_enhance_ms":round(t_enh,1),"T_depth_ms":round(t_inf,1),
                    "T_total_ms":round(t_enh+t_inf,1),
                    "FPS":round(1000/(t_enh+t_inf+1e-6),1),
                    **m,
                })
                n_new += 1
                if n_new % CKPT_EVERY == 0:
                    pd.DataFrame(results).to_csv(CKPT_CSV, index=False)
    df = pd.DataFrame(results)
    df.to_csv(CKPT_CSV, index=False)
    print(f"\nExperimento completo — {len(df)} evaluaciones ({n_new} nuevas)")


---
## 7. Resumen de resultados

## 6. Resumen de resultados

Se agregan los resultados por **(modelo, método)** promediando sobre los 550 fotogramas, y se calcula la **mejora porcentual de AbsRel** de cada *enhancement* respecto a su línea base (`none`). Un valor **negativo** (marcado con ←) indica que el realce **mejoró** la profundidad.

In [ ]:
numeric_cols = ["AbsRel","SqRel","RMSE","RMSELog",
                "delta_1","delta_2","delta_3",
                "AbsRel_spec","AbsRel_nospec","T_total_ms","FPS"]

summary = (df.groupby(["Modelo","Método"])[numeric_cols]
             .mean().round(4)
             .sort_values(["Modelo","AbsRel"]))

for model in df["Modelo"].unique():
    sub = summary.loc[model]
    if "none" in sub.index:
        b = sub.loc["none","AbsRel"]
        summary.loc[(model,slice(None)),"ΔAbsRel_%"] = (
            (summary.loc[(model,slice(None)),"AbsRel"]-b)/b*100).round(1)

# Compatibilidad
if "FPS" not in summary.columns:
    summary["FPS"] = df.groupby(["Modelo","Método"])["FPS"].mean().round(1)

print("="*115)
print(f"{'Modelo':<18} {'Enhancement':<12} {'AbsRel':>7} {'SqRel':>7} {'RMSE':>7} "
      f"{'RMSELog':>8} {'d1.25':>6} {'d1.25^2':>8} {'d1.25^3':>8} "
      f"{'IT(ms)':>7} {'ΔAR%':>6}")
print("-"*115)
for model in df["Modelo"].unique():
    for method in ["none","retinex","endolmspec","iat","endosttn"]:
        if (model,method) not in summary.index: continue
        row = summary.loc[(model,method)]
        delta = f"{row['ΔAbsRel_%']:+.1f}%" if method!="none" else "base"
        mark  = " ←" if method!="none" and row["AbsRel"]<summary.loc[(model,"none"),"AbsRel"] else ""
        print(f"  {model:<16} {method:<12} "
              f"{row['AbsRel']:>7.4f} {row['SqRel']:>7.4f} {row['RMSE']:>7.3f} "
              f"{row['RMSELog']:>8.4f} {row['delta_1']:>6.4f} {row['delta_2']:>8.4f} "
              f"{row['delta_3']:>8.4f} {row['T_total_ms']:>7.1f} {delta:>6}{mark}")
    print()
print("ΔAR% negativo = mejora vs. baseline (none)")


---
## 8. Visualizaciones

In [ ]:
import matplotlib.pyplot as plt

models_list  = list(df["Modelo"].unique())
methods_list = list(df["Método"].unique())
colors = {"none":"#8aa0c0","retinex":"#4599EC","endolmspec":"#E0A800","iat":"#2ecc71","endosttn":"#9b59b6"}
_DEF_COLOR = "#95a5a6"  # gris para cualquier metodo no listado
x = np.arange(len(methods_list))

metrics_plot = [
    ("AbsRel",      "AbsRel (↓ mejor)",           "Primaria"),
    ("RMSE",        "RMSE mm (↓ mejor)",            "Geométrica"),
    ("AbsRel_spec", "AbsRel especular (↓ mejor)",   "Hipótesis"),
    ("FPS",         "FPS total (↑ mejor)",           "Clínica"),
]
n_m, n_met = len(models_list), len(metrics_plot)

fig, axes = plt.subplots(n_m, n_met, figsize=(4*n_met, 4*n_m))
fig.suptitle("Avance 5 — Enhancement × Modelos SCARED\n(promedio split oficial: 550 frames, ds 1-7)",
             fontsize=13, fontweight="bold")

for r, model in enumerate(models_list):
    sub = summary.loc[model] if model in summary.index.get_level_values(0) else None
    for c, (metric, label, cat) in enumerate(metrics_plot):
        ax = axes[r,c] if n_m>1 else axes[c]
        vals = [sub.loc[m,metric] if sub is not None and m in sub.index else np.nan
                for m in methods_list]
        bars = ax.bar(x, vals, 0.62, color=[colors.get(m,_DEF_COLOR) for m in methods_list])
        ax.set_xticks(x); ax.set_xticklabels(methods_list, fontsize=9)
        ax.grid(axis="y", alpha=0.3)
        if r==0: ax.set_title(f"{label}\n[{cat}]", fontsize=9)
        if c==0: ax.set_ylabel(model, fontsize=10, fontweight="bold")
        top = max(v for v in vals if not np.isnan(v)) if any(not np.isnan(v) for v in vals) else 1
        for b,v in zip(bars,vals):
            if not np.isnan(v):
                ax.text(b.get_x()+b.get_width()/2, b.get_height()+top*0.015,
                        f"{v:.3f}" if metric!="FPS" else f"{v:.0f}",
                        ha="center", va="bottom", fontsize=7)
        ax.set_ylim(0, top*1.22)

plt.tight_layout(rect=[0,0,1,0.96])
plt.savefig(OUT_DIR/"avance5_metricas.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Visualización cualitativa

Para inspeccionar visualmente, se elige un fotograma representativo de dos *keyframes* distintos del *split* y se muestran, lado a lado, los mapas de profundidad predichos por cada modelo bajo cada *enhancement*, junto al *ground-truth*. El mapa de color `plasma_r` codifica **amarillo = cerca, azul = lejos**, normalizado por percentiles (p2–p98) para que los detalles sean visibles.

In [ ]:
# Un frame representativo de dos keyframes distintos del split
_kf_list = list(SPLIT_BY_KF.keys())
VIZ_FRAMES = [(_kf_list[0][0], _kf_list[0][1], SPLIT_BY_KF[_kf_list[0]][0]),
              (_kf_list[len(_kf_list)//2][0], _kf_list[len(_kf_list)//2][1], SPLIT_BY_KF[_kf_list[len(_kf_list)//2]][0])]
CMAP = "plasma_r"

for ds_id, kf_id, fid in VIZ_FRAMES:
    img_rgb, gt_mm = load_split_frame(ds_id, kf_id, fid)
    valid_gt = (~np.isnan(gt_mm))&(gt_mm>0)&(gt_mm<CAP_MM)
    corr_names = list(CORRECTIONS.keys())
    n_cols = 1+len(corr_names)+1
    fig, axes = plt.subplots(len(DEPTH_MODELS), n_cols,
                             figsize=(3.5*n_cols, 3.5*len(DEPTH_MODELS)))

    cache = {}
    for model_name, depth_fn in DEPTH_MODELS.items():
        for cn, cf in CORRECTIONS.items():
            ic = cf(img_rgb, ds=ds_id, kf=kf_id, fid=fid) if cn=='endosttn' else cf(img_rgb); dr, _ = depth_fn(ic)
            sc = np.median(gt_mm[valid_gt])/(np.median(dr[valid_gt])+1e-8)
            cache[(model_name,cn)] = (ic, dr, dr*sc)

    for row,(model_name,_) in enumerate(DEPTH_MODELS.items()):
        ax_row = axes[row]
        ax_row[0].imshow(img_rgb); ax_row[0].axis("off")
        ax_row[0].set_ylabel(model_name, fontsize=8, fontweight="bold")
        if row==0: ax_row[0].set_title("RGB", fontsize=8)

        for col, cn in enumerate(corr_names, 1):
            ic, dr, pm = cache[(model_name,cn)]
            m = compute_metrics(img_rgb, ic, dr, gt_mm, CAP_MM)
            v1,v2 = np.nanpercentile(pm[valid_gt],2), np.nanpercentile(pm[valid_gt],98)
            ax_row[col].imshow(pm, cmap=CMAP, vmin=v1, vmax=v2); ax_row[col].axis("off")
            if row==0: ax_row[col].set_title(f"{cn}\nAbsRel={m['AbsRel']:.4f}", fontsize=7)
            else:      ax_row[col].set_title(f"AbsRel={m['AbsRel']:.4f}", fontsize=7)

        v1,v2 = np.nanpercentile(gt_mm[valid_gt],2), np.nanpercentile(gt_mm[valid_gt],98)
        ax_row[-1].imshow(gt_mm, cmap=CMAP, vmin=v1, vmax=v2); ax_row[-1].axis("off")
        if row==0: ax_row[-1].set_title("GT (mm)", fontsize=8)

    plt.suptitle(f"Avance 5 NV — {ds_id}/{kf_id} f{fid}  (plasma_r: amarillo=cerca, azul=lejos)",
                 fontsize=9, y=1.01)
    plt.tight_layout()
    plt.savefig(OUT_DIR/f"avance5nv_viz_{ds_id}_{kf_id}_{fid}.png", dpi=110, bbox_inches="tight")
    plt.show()


### 8b. Efecto visual de Endo-STTN sobre la imagen

A diferencia de los demas *enhancements*, Endo-STTN no ajusta brillo/contraste sino que **elimina los reflejos especulares** mediante *inpainting* temporal: detecta las zonas de brillo especular (mascara) y reconstruye el tejido oculto usando los fotogramas vecinos. La siguiente figura muestra, para fotogramas representativos, la imagen original, la mascara de especulares detectada y el resultado corregido (con un *zoom* a una zona de reflejo).

In [ ]:
# Efecto visual de Endo-STTN: original -> mascara de especulares -> imagen corregida
# Endo-STTN rellena (inpainting) los reflejos especulares usando frames vecinos.
if _sttn_ok:
    import matplotlib.pyplot as plt
    for ds_id, kf_id, fid in VIZ_FRAMES:
        img_rgb, _ = load_split_frame(ds_id, kf_id, fid)
        mask = _sttn_specular_mask(img_rgb)                 # mascara binaria de reflejos
        img_sttn = correct_endo_sttn(img_rgb, ds=ds_id, kf=kf_id, fid=fid)

        # zoom a la zona con mas especulares (centroide de la mascara)
        ys, xs = np.where(mask > 0)
        if len(xs) > 0:
            cy, cx = int(ys.mean()), int(xs.mean())
            half = 110
            H, W = img_rgb.shape[:2]
            y0, y1 = max(0, cy-half), min(H, cy+half)
            x0, x1 = max(0, cx-half), min(W, cx+half)
        else:
            y0, y1, x0, x1 = 0, img_rgb.shape[0], 0, img_rgb.shape[1]

        fig, axes = plt.subplots(2, 3, figsize=(13, 8.5))
        # fila 1: imagen completa
        axes[0,0].imshow(img_rgb);              axes[0,0].set_title("RGB original", fontsize=10)
        axes[0,1].imshow(mask, cmap="gray");    axes[0,1].set_title("Mascara de especulares", fontsize=10)
        axes[0,2].imshow(img_sttn);             axes[0,2].set_title("Tras Endo-STTN (inpainting)", fontsize=10)
        # fila 2: zoom a la zona de reflejo
        axes[1,0].imshow(img_rgb[y0:y1, x0:x1]);   axes[1,0].set_title("Zoom: original", fontsize=10)
        axes[1,1].imshow(mask[y0:y1, x0:x1], cmap="gray"); axes[1,1].set_title("Zoom: mascara", fontsize=10)
        axes[1,2].imshow(img_sttn[y0:y1, x0:x1]);  axes[1,2].set_title("Zoom: corregida", fontsize=10)
        for ax in axes.ravel(): ax.axis("off")

        plt.suptitle(f"Endo-STTN — eliminacion de reflejos especulares  ({ds_id}/{kf_id} f{fid})",
                     fontsize=12, fontweight="bold", y=1.0)
        plt.tight_layout()
        plt.savefig(OUT_DIR/f"avance5nv_sttn_effect_{ds_id}_{kf_id}_{fid}.png", dpi=120, bbox_inches="tight")
        plt.show()
        cov = mask.mean()*100
        print(f"{ds_id}/{kf_id} f{fid}: {cov:.1f}% de pixeles marcados como especulares y rellenados")
else:
    print("Endo-STTN no disponible (pesos no cargados): se omite la visualizacion del efecto.")


---
## 9. Resultados sobre el *split* oficial (550 frames, datasets 1--7)

Esta es la tabla principal del Avance 5 — New Version: 6 modelos re-entrenados en SCARED × 5 *enhancements*, evaluados sobre el *split* oficial de AF-SfMLearner.

### 9.1 Baseline (sin enhancement) — ranking de modelos

| Modelo | AbsRel | RMSE (mm) | δ<1.25 |
|---|---|---|---|
| **MonoIIT** | **0.0554** | **4.700** | 0.9713 |
| **w19-MonoViT** | 0.0727 | 5.690 | 0.9516 |
| **Monodepth2** | 0.0708 | 5.746 | 0.9459 |
| **MonoViT** | 0.0994 | 8.537 | 0.8908 |
| **EndoSfMLearner** | 0.1123 | 8.994 | 0.8686 |
| **AF-SfMLearner** | 0.1177 | 9.093 | 0.8476 |

> **MonoIIT (AbsRel 0.0554) es el mejor modelo** sobre el split oficial.

### 9.2 Efecto del enhancement por modelo (ΔAbsRel% vs. *none*)

| Modelo | retinex | endolmspec | iat | endosttn | Mejor |
|---|---|---|---|---|---|
| Monodepth2     | +54.7% | +4.9% | **−1.7%** ← | +1.7% | iat |
| MonoViT        | −0.8% ← | +4.1% | **−4.1%** ← | −2.3% ← | iat |
| EndoSfMLearner | +0.3% | +0.1% | +0.1% | +0.0% | (sin efecto) |
| AF-SfMLearner  | −12.9% ← | −9.2% ← | **−13.9%** ← | −0.8% ← | iat |
| MonoIIT        | +53.6% | +1.4% | **−3.1%** ← | +0.7% | iat |
| w19-MonoViT    | +18.0% | +0.6% | **−1.9%** ← | +0.1% | iat |

> ← = mejora respecto al baseline. Negativo = mejor.

### Hallazgos

1. **IAT es el enhancement más consistente:** mejora en **5 de 6 modelos**, y es el mejor método en todos los que se benefician. Es el único realce que vale la pena de forma general.
2. **AF-SfMLearner es el gran beneficiado del enhancement:** IAT le da **−13.9%**, retinex −12.9%, endolmspec −9.2%. Es el modelo donde la corrección de imagen tiene más impacto — coherente con que su *baseline* es el más débil (AbsRel 0.1177).
3. **Retinex castiga a los modelos fuertes:** +54.7% en Monodepth2, +53.6% en MonoIIT, +18.0% en w19-MonoViT. La corrección clásica distorsiona entradas que el modelo ya maneja bien.
4. **Endo-STTN: mejoras marginales a un costo enorme.** Su efecto va de +1.7% a −2.3% (apenas mejora en MonoViT/AF-SfMLearner), pero su tiempo de inferencia es **~7,700 ms/frame**, unas **30× más lento** que IAT (~250 ms) por su naturaleza temporal (ventana de 10 frames). No es competitivo como *enhancement* general, aunque su corrección de especulares podría justificarse en un análisis 3D dedicado (ver celda Chamfer opcional).
5. **EndoSfMLearner es insensible al enhancement** (todos los métodos ≈ 0%): su arquitectura DispResNet a baja resolución no aprovecha los cambios de la imagen de entrada.

> **Comparación A4 vs. A5 sobre este split:** queda pendiente re-correr el Avance 4 (pesos del paper) con el mismo *split* oficial para una comparación A4 vs. A5 estrictamente válida. Las tablas comparativas previas usaban el protocolo anterior (10 keyframes, datasets 8--9) y **no** son directamente comparables con estos números.

In [ ]:
# Cargar resultados de Avance 4 si existen
a4_csv = REPO_ROOT / "outcomes" / "avance4" / "avance4_results.csv"
if a4_csv.exists():
    df4 = pd.read_csv(a4_csv)
    base4 = (df4[df4["Método"]=="none"]
               .groupby("Modelo")[["AbsRel","RMSE","delta_1","FPS"]]
               .mean().round(4))
    base4.columns = [f"{c}_A4" for c in base4.columns]

    base5 = (df[df["Método"]=="none"]
               .groupby("Modelo")[["AbsRel","RMSE","delta_1","FPS"]]
               .mean().round(4))
    base5.columns = [f"{c}_A5" for c in base5.columns]

    comp = base4.join(base5, how="outer")
    # Delta: A5 vs A4
    for col in ["AbsRel","RMSE"]:
        comp[f"Δ{col}"] = ((comp[f"{col}_A5"]-comp[f"{col}_A4"])/comp[f"{col}_A4"]*100).round(1)

    print("Comparativa baseline (none): A4 (pesos originales) vs A5 (pesos SCARED)")
    print("Δ negativo = A5 es mejor\n")
    print(comp.to_string())
else:
    print(f"No se encontró {a4_csv}")
    print("Ejecuta primero el Avance 4 para tener resultados comparativos.")


---
## 10. Exportar resultados al repositorio

Las imágenes se guardan en `outcomes/avance5/`.


In [ ]:
import subprocess, shutil

def git(args):
    subprocess.check_call(["git","-C",str(REPO_ROOT)]+args)

if IN_COLAB:
    from getpass import getpass
    token = getpass("GitHub token (repo scope): ")
    git(["remote","set-url","origin",
         f"https://{token}@github.com/jmtoral/proyecto_integrador_52.git"])
    git(["pull","--rebase"])

new_files = list(OUT_DIR.glob("avance5_*.png"))
csv_file  = OUT_DIR/"avance5_results.csv"
if csv_file.exists(): new_files.append(csv_file)

print(f"OUT_DIR  : {OUT_DIR}")
print(f"Archivos : {[f.name for f in new_files]}")

if new_files:
    git(["add"]+[str(f.relative_to(REPO_ROOT)) for f in new_files])
    ch = subprocess.run(["git","-C",str(REPO_ROOT),"diff","--cached","--name-only"],
                        capture_output=True,text=True).stdout.strip()
    if ch:
        git(["commit","-m",f"Add: resultados Avance5 ({len(new_files)} archivos)"])
        git(["push"])
        print(f"Push OK: {[f.name for f in new_files]}")
    else:
        print("Sin cambios nuevos")


## 5b. (Opcional para publicación) Chamfer 3D: baseline vs. Endo-STTN

Chamfer es una métrica de **calidad geométrica 3D** (distancia entre la nube de puntos predicha y la del *ground-truth*). **No** se incluye en el bucle principal porque es cara (KD-Trees en CPU) y no forma parte de la tabla estándar de profundidad.

Pero para un artículo (p. ej. **MICAI/MICCAI**), donde el aporte es que **Endo-STTN** mejora la reconstrucción de la superficie, una métrica 3D refuerza el argumento. Esta celda calcula Chamfer **solo** para la comparación relevante —`none` (baseline) vs. `endosttn`— sobre los 6 modelos, en vez de las 30 combinaciones. Así corre en **minutos**, no horas.

> Es **independiente** del experimento principal: se ejecuta únicamente si se requiere la métrica geométrica 3D (p. ej. para una publicación).

In [ ]:
# Chamfer 3D solo para baseline (none) vs Endo-STTN — uso en publicacion.
# Independiente del loop principal. Ejecutar bajo demanda.
import pandas as pd
from tqdm import tqdm

PAPER_METHODS = ["none", "endosttn"]   # comparacion clave para el paper
CHAMFER_CSV = OUT_DIR / "chamfer_paper_none_vs_endosttn.csv"

def _apply_corr_paper(name, img_rgb, ds, kf, fid):
    fn = CORRECTIONS[name]
    if name == "endosttn":
        return fn(img_rgb, ds=ds, kf=kf, fid=fid)
    return fn(img_rgb)

ch_rows = []
for (ds, kf, fid) in tqdm(SPLIT_ITEMS, desc="Chamfer (none vs endosttn)"):
    img_rgb, gt_mm = load_split_frame(ds, kf, fid)
    if gt_mm is None:
        continue
    valid = (~np.isnan(gt_mm)) & (gt_mm > 0) & (gt_mm < CAP_MM)
    if valid.sum() == 0:
        continue
    pc_gt = depth_to_pc(np.where(valid, gt_mm, np.nan), valid)
    for corr_name in PAPER_METHODS:
        if corr_name not in CORRECTIONS:
            continue
        img_corr = _apply_corr_paper(corr_name, img_rgb, ds, kf, fid)
        for model_name, depth_fn in DEPTH_MODELS.items():
            depth_rel, _ = depth_fn(img_corr)
            scale = np.median(gt_mm[valid]) / (np.median(depth_rel[valid]) + 1e-8)
            pred = depth_rel * scale
            pc_pred = depth_to_pc(np.where(valid, pred, np.nan), valid)
            ch = chamfer(pc_pred, pc_gt)
            ch_rows.append({"Dataset": ds, "Keyframe": kf, "Frame": fid,
                            "Modelo": model_name, "Metodo": corr_name, "Chamfer": ch})

df_chamfer = pd.DataFrame(ch_rows)
df_chamfer.to_csv(CHAMFER_CSV, index=False)

# Resumen: Chamfer medio por modelo y metodo + mejora de endosttn vs none
print("Chamfer 3D medio (mm) — menor es mejor\n" + "="*52)
piv = df_chamfer.groupby(["Modelo", "Metodo"])["Chamfer"].mean().unstack()
for model in piv.index:
    base = piv.loc[model, "none"] if "none" in piv.columns else float("nan")
    sttn = piv.loc[model, "endosttn"] if "endosttn" in piv.columns else float("nan")
    delta = (sttn - base) / base * 100 if base == base and base != 0 else float("nan")
    mark = "  <- mejora" if delta == delta and delta < 0 else ""
    print(f"  {model:16s} none={base:7.3f}  endosttn={sttn:7.3f}  ({delta:+.1f}%){mark}")
print(f"\nGuardado: {CHAMFER_CSV.name}")
